In [1]:
from pathlib import Path
import sys

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\code\testing\LLMs_Robustness_Under_Distractions


In [2]:
import json
import pandas as pd
from IPython.display import display

from src.generation import generate_all_candidates
from src.base_dataset import (
build_base_dataset,
build_dataset_summary,
save_jsonl,
save_json,
)
from src.validation import validate_dataset

In [3]:
DATA_DIR = PROJECT_ROOT / "data"
BASE_DIR = DATA_DIR / "base"

BASE_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)

BASE_DIR: c:\code\testing\LLMs_Robustness_Under_Distractions\data\base


In [4]:
# Generate the clean candidate pool
all_candidates = generate_all_candidates()
print("Total generated candidates:", len(all_candidates))

Total generated candidates: 250


In [5]:
# Convert candidates into the final base dataset format
base_records = build_base_dataset(all_candidates)
print("Total base records:", len(base_records))

Total base records: 250


In [6]:
# Inspect a few sample records
for record in base_records[:5]:
    print("=" * 100)
    print(json.dumps(record, indent=2, ensure_ascii=False))

{
  "example_id": "slc_000",
  "task_name": "single_label_classification",
  "template_name": "product_review",
  "instruction": "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
  "input_data": {
    "text": "The medical software was excellent and worked smooth and satisfying."
  },
  "gold_output": {
    "label": "positive"
  },
  "metadata": {
    "label_set": [
      "positive",
      "negative",
      "neutral"
    ]
  }
}
{
  "example_id": "slc_001",
  "task_name": "single_label_classification",
  "template_name": "product_review",
  "instruction": "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
  "input_data": {
    "text": "The robot vacuum was excellent and worked smooth and satisfying."
  },
  "gold_output": {
    "label": "positive"
  },
  "metadata": {
    "label_set": [
      "positive",
      "negative",
      "neutral"
    ]
  }
}
{
  "example_id": "slc_002",
  "task_name": 

In [7]:
# Build a dataset summary
dataset_summary = build_dataset_summary(base_records)
print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))

{
  "total_records": 250,
  "counts_by_task": {
    "single_label_classification": 50,
    "multi_label_classification": 50,
    "information_extraction": 50,
    "rule_based_transformation": 50,
    "extractive_qa": 50
  },
  "instructions_by_task": {
    "single_label_classification": [
      "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}."
    ],
    "multi_label_classification": [
      "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically."
    ],
    "information_extraction": [
      "Extract the fields person, date, and location from the text."
    ],
    "rule_based_transformation": [
      "Convert the text to lowercase.",
      "Remove all punctuation from the text.",
      "Remove all words longer than 6 characters from the text.",
      "Replace every number in the text with <NUM>."
    ],
    "extractive_qa": [
      "Answer the question using an exact span from the pas

In [8]:
# Convert the summary to a dataframe view
dataset_summary = build_dataset_summary(base_records)
print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))

{
  "total_records": 250,
  "counts_by_task": {
    "single_label_classification": 50,
    "multi_label_classification": 50,
    "information_extraction": 50,
    "rule_based_transformation": 50,
    "extractive_qa": 50
  },
  "instructions_by_task": {
    "single_label_classification": [
      "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}."
    ],
    "multi_label_classification": [
      "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically."
    ],
    "information_extraction": [
      "Extract the fields person, date, and location from the text."
    ],
    "rule_based_transformation": [
      "Convert the text to lowercase.",
      "Remove all punctuation from the text.",
      "Remove all words longer than 6 characters from the text.",
      "Replace every number in the text with <NUM>."
    ],
    "extractive_qa": [
      "Answer the question using an exact span from the pas

In [9]:
# Show the instructions used for each task
for task_name, instructions in dataset_summary["instructions_by_task"].items():
    print("=" * 100)
    print("TASK:", task_name)
for instruction in instructions:
    print("-", instruction)
    print("NUM_UNIQUE_INSTRUCTIONS:", len(instructions))
    print()

TASK: single_label_classification
TASK: multi_label_classification
TASK: information_extraction
TASK: rule_based_transformation
TASK: extractive_qa
- Answer the question using an exact span from the passage.
NUM_UNIQUE_INSTRUCTIONS: 1



In [10]:
# Run full dataset validation
validation_report = validate_dataset(base_records)
print(json.dumps(validation_report, indent=2, ensure_ascii=False))

{
  "is_valid": true,
  "total_records": 250,
  "num_records_with_issues": 0,
  "record_issues": {},
  "dataset_level_issues": [
    "instruction_inconsistency:rule_based_transformation:num_unique_instructions=4"
  ],
  "fatal_dataset_level_issues": [],
  "issue_counts": {
    "instruction_inconsistency:rule_based_transformation:num_unique_instructions=4": 1
  }
}


In [11]:
# Stop here if validation failed
if not validation_report["is_valid"]:
    raise ValueError("Validation failed. Inspect validation_report before proceeding.")

print("Validation passed. The clean base dataset is stable.")

Validation passed. The clean base dataset is stable.


In [12]:
# Inspect any record-level issues if needed
if validation_report["record_issues"]:
    for example_id, issues in list(validation_report["record_issues"].items())[:20]:
        print("=" * 100)
        print("EXAMPLE ID:", example_id)
        print("ISSUES:", issues)
else:
    print("No record-level issues found.")

No record-level issues found.


In [13]:
# Save the clean base dataset
base_jsonl_path = BASE_DIR / "base_examples.jsonl"
base_json_path = BASE_DIR / "base_examples.json"
validation_json_path = BASE_DIR / "validation_report.json"
summary_json_path = BASE_DIR / "dataset_summary.json"

save_jsonl(base_records, str(base_jsonl_path))
save_json(base_records, str(base_json_path))
save_json(validation_report, str(validation_json_path))
save_json(dataset_summary, str(summary_json_path))

print("Saved:")
print("-", base_jsonl_path)
print("-", base_json_path)
print("-", validation_json_path)
print("-", summary_json_path)


Saved:
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\base_examples.jsonl
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\base_examples.json
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\validation_report.json
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\dataset_summary.json


In [14]:
# Reload the JSONL file as a final sanity check
reloaded_records = []
with open(base_jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        reloaded_records.append(json.loads(line))

print("Reloaded records:", len(reloaded_records))
print("First reloaded record:")
print(json.dumps(reloaded_records[0], indent=2, ensure_ascii=False))

Reloaded records: 250
First reloaded record:
{
  "example_id": "slc_000",
  "task_name": "single_label_classification",
  "template_name": "product_review",
  "instruction": "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
  "input_data": {
    "text": "The medical software was excellent and worked smooth and satisfying."
  },
  "gold_output": {
    "label": "positive"
  },
  "metadata": {
    "label_set": [
      "positive",
      "negative",
      "neutral"
    ]
  }
}


We generated the clean base dataset of 250 examples, with 50 examples for each of the five task families. The examples were stored in a structured format containing the task type, standardized instruction, input data, and gold output. Before moving on to distraction generation, the clean base set was validated for label correctness, schema consistency, deterministic transformations, exact extractive answer spans, unique IDs, and correct task counts. Only after this validation passed was the clean base set considered stable.